# Research evidence review

Report-only notebook: reads recorded aggregate validation results. It never trains a model or submits predictions. Historical validation is not the 2026 leaderboard. Run from this repository with the existing Python (March Mania) kernel. The implementation archives and protocols are linked in `research/README.md`.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
pio.renderers.default = 'plotly_mimetype'
base = Path.cwd().resolve()
repo = next((p for p in [base, *base.parents] if (p/'portfolio/validation_metrics.csv').is_file()), None)
assert repo is not None, 'Open this notebook inside the original repository clone.'
df = pd.read_csv(repo/'portfolio/validation_metrics.csv', dtype={'round':str})
assert df['brier'].between(0,1).all()
display(df.head(10))


## Within-round comparisons
Arms are historical labels, not rankings across different validation populations. Compare each arm only with its matching reference.


In [ ]:
avg=df.groupby(['round','population','arm'],as_index=False)['brier'].mean()
fig=px.scatter(avg,x='round',y='brier',color='population',symbol='arm',hover_data=['arm'],title='Exploratory mean-season Brier by research round',labels={'brier':'Mean-season Brier (lower is better)','round':'Research round'})
fig.show()
refs=df[df.arm=='Reference'][['round','population','season','brier']].rename(columns={'brier':'reference_brier'})
paired=df.merge(refs,on=['round','population','season'],validate='many_to_one')
paired['delta']=paired.brier-paired.reference_brier
fig=px.box(paired[paired.arm!='Reference'],x='round',y='delta',color='population',points='all',hover_data=['season','arm'],title='Within-round Brier changes by season',labels={'delta':'Brier change vs matching reference','round':'Research round'})
fig.add_hline(y=0)
fig.show()


## Evaluation coverage
More comparisons are not proof of better predictions. Seasons are reused exploratory history; no untouched test is claimed.


In [ ]:
coverage=df.groupby(['round','population'],as_index=False).agg(seasons=('season','nunique'),comparisons=('arm','size'))
fig=px.bar(coverage,x='round',y='comparisons',color='population',barmode='group',hover_data=['seasons'],title='Reported comparisons and season coverage',labels={'comparisons':'Recorded comparison rows','round':'Research round'})
fig.show()
display(coverage)
